## Week 13 Practice

### 13-1

In [20]:
import torch 
import torch.nn.functional as F

In [21]:
# maps words to vectors
em = torch.nn.Embedding(50, embedding_dim = 16)
em(torch.tensor(0))

tensor([ 0.3506,  0.6095, -0.6251,  0.1401, -0.4354,  0.3768,  0.3675, -2.5915,
        -0.5707,  0.7122,  0.7513, -0.1406,  1.9130,  0.7866,  0.7848, -0.1519],
       grad_fn=<EmbeddingBackward0>)

In [22]:
def attn(Q, K, V): 
    # Q = query vector (words) [d]; scaling factor so gradients don't get too large
    # K = key vectors (words) [n x d]
    # V = value vectors (words) [n x v]
    d = Q.shape[-1] 
    scores = (Q @ K.T) / (d ** 0.5)
    probs = torch.softmax(scores, 0) # [n]
    output = probs @ V # [n], [n, d] -> [d]
    return output, probs

d = 16
n = 50
Q = torch.randn((d,))
K = torch.randn((n, d))
V = torch.randn((n, d))
attn(Q, K, V)

(tensor([ 0.4378, -0.0305, -0.6129,  0.0300, -0.2369,  0.5571,  0.1727, -0.7293,
          0.3217,  0.5087,  0.6400,  0.1105, -0.1558, -0.0501,  0.5109,  0.2345]),
 tensor([1.8554e-03, 2.2695e-02, 1.8984e-03, 2.1704e-03, 3.0509e-02, 2.4416e-03,
         4.8405e-03, 1.3780e-03, 8.0135e-03, 1.1162e-02, 1.5455e-02, 1.0451e-02,
         1.8302e-04, 1.4236e-03, 8.7992e-03, 2.6773e-02, 1.9902e-02, 5.4209e-02,
         8.5411e-03, 4.1694e-03, 4.3270e-03, 1.8209e-03, 2.6680e-02, 5.7804e-04,
         1.5911e-02, 9.4042e-03, 1.6441e-02, 5.1953e-04, 6.8341e-03, 9.6541e-02,
         9.0282e-03, 5.5129e-02, 5.6136e-04, 2.1961e-03, 2.0716e-02, 2.5370e-03,
         6.3442e-04, 4.6555e-03, 9.9939e-03, 1.3183e-02, 8.3558e-04, 5.8539e-03,
         3.8284e-01, 3.7155e-02, 7.4473e-03, 5.9490e-03, 4.0917e-03, 5.5483e-03,
         2.1438e-03, 1.3569e-02]))

In [23]:
def self_attn(X): 
    return attn(X, X, X)

In [24]:
em = torch.nn.Embedding(50, embedding_dim = 16)

'hello my name is' 
# random vector representations for 5 words
# [32, 8, 7, 14, 17]

X = em(torch.tensor([32, 8, 7, 14, 17])) # [5, 16]

In [25]:
self_attn(X)[0] # [5, 16]
self_attn(X)[1] # [5, 5]

tensor([[0.7517, 0.0296, 0.0247, 0.0218, 0.0041],
        [0.1038, 0.9117, 0.0401, 0.0200, 0.0024],
        [0.1085, 0.0502, 0.9295, 0.0107, 0.0047],
        [0.0257, 0.0067, 0.0029, 0.9397, 0.0037],
        [0.0103, 0.0017, 0.0027, 0.0078, 0.9851]], grad_fn=<SoftmaxBackward0>)

In [26]:
class SelfAttention(torch.nn.Module): 
    def __init__(self, d): 
        super().__init__()
        self.W_Q = torch.nn.Linear(d, d, bias=False)
        self.W_K = torch.nn.Linear(d, d, bias=False)
        self.W_V = torch.nn.Linear(d, d, bias=False)

    def forward(self, X): 
        Q = self.W_Q(X)
        K = self.W_K(X)
        V = self.W_V(X)
        return attn(Q, K, V)

In [27]:
sa = SelfAttention(d=16)
sa

SelfAttention(
  (W_Q): Linear(in_features=16, out_features=16, bias=False)
  (W_K): Linear(in_features=16, out_features=16, bias=False)
  (W_V): Linear(in_features=16, out_features=16, bias=False)
)

In [ ]:
class MHSA(torch.nn.Module): 
    def __init__(self, d, num_heads): 
        super().__init__() 
        assert d % num_heads == 0 # ensure divisibility
        self.d_head = d // num_heads
        self.num_heads = num_heads
        self.W_Q = torch.nn.Linear(d, d, bias=False)
        self.W_K = torch.nn.Linear(d, d, bias=False)
        self.W_V = torch.nn.Linear(d, d, bias=False)
        self.W_O = torch.nn.Linear(d, d, bias=False)

    def forward(self, X): 
        n, d_model = X.shape 
        Q = self.W_Q(X) #[n, d]
        K = self.W_K(X) #[n, d]
        V = self.W_V(X) #[n, d]
        Q = Q.reshape(n, self.num_heads, self.d_head) #[n, h, d_head = d/h]
        K = K.reshape(n, self.num_heads, self.d_head) #[n, h, d_head = d/h]
        V = V.reshape(n, self.num_heads, self.d_head) #[n, h, d_head = d/h]

        scores = (Q @ K.transpose(1,2)) / (d ** 0.5)
        weights = torch.nn.softmax(scores, dim=-1) #[n, h, n]
        out = weights @ V
        return scores, out 


In [38]:
Qr = Q.reshape(2, 8) # (h, d/h)
Kr = K.reshape((50, 2, 8)) # (n, h, d/h)

Kr.transpose(1, 2).shape
(Qr @ Kr.transpose(1,2)).shape
torch.softmax((Qr @ Kr.transpose(1, 2)), dim = -1)

tensor([[[9.8890e-01, 1.1100e-02],
         [5.9217e-01, 4.0783e-01]],

        [[1.4096e-01, 8.5904e-01],
         [6.6875e-02, 9.3313e-01]],

        [[2.0291e-02, 9.7971e-01],
         [8.8017e-01, 1.1983e-01]],

        [[8.8774e-02, 9.1123e-01],
         [6.1031e-02, 9.3897e-01]],

        [[9.3020e-01, 6.9800e-02],
         [4.8655e-01, 5.1345e-01]],

        [[1.3445e-02, 9.8656e-01],
         [1.2318e-01, 8.7682e-01]],

        [[6.1246e-01, 3.8754e-01],
         [1.1177e-01, 8.8823e-01]],

        [[4.2573e-01, 5.7427e-01],
         [5.4012e-01, 4.5988e-01]],

        [[7.4087e-01, 2.5913e-01],
         [9.8412e-01, 1.5881e-02]],

        [[1.1796e-03, 9.9882e-01],
         [6.6945e-02, 9.3305e-01]],

        [[5.6002e-01, 4.3998e-01],
         [1.0404e-02, 9.8960e-01]],

        [[1.2596e-02, 9.8740e-01],
         [1.0310e-01, 8.9690e-01]],

        [[1.5484e-04, 9.9985e-01],
         [8.7122e-01, 1.2878e-01]],

        [[9.0987e-02, 9.0901e-01],
         [1.7279e-01, 8.2721e

In [40]:
print(torch.nn.MultiheadAttention) # built-in module
print(torch.nn.MultiheadAttention(16, 16))

<class 'torch.nn.modules.activation.MultiheadAttention'>
MultiheadAttention(
  (out_proj): NonDynamicallyQuantizableLinear(in_features=16, out_features=16, bias=True)
)
